# Calibration 04-3 — Nelder-Mead vs L-BFGS-B 비교

이전에 실행한 두 fit 결과 (04-1, 04-2) 를 JSON 에서 로드해 직접 비교.

**입력**:
- `outputs/calibration/2019-2020_by_age_NM.json`
- `outputs/calibration/2019-2020_by_age_LBFGS.json`

**비교 항목**:
1. NLL / evaluations / elapsed time
2. 핵심 파라미터 (β 4 채널, γ_report, amp, base, sigma)
3. φ_a 연령별 분포
4. 7 연령 그룹 fit (side-by-side overlay)
5. 결론 — 어느 방법이 더 좋은가

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

from kt_data import ILI_AGE_GROUPS
from kt_epimodel.calibration.ili_target import (
    load_ili_target_by_age, simulation_to_ili_by_age,
)
from kt_epimodel.calibration.optimizer import load_result
from kt_epimodel.calibration.simple_model import (
    build_aggregated_inputs, simulate_aggregated,
    estimate_initial_infected_from_ili,
)
from kt_epimodel.model.compartments import IDX_S
from kt_epimodel.model.parameters import (
    DiseaseParameters, ModelParameters,
)

OUT = Path('../outputs/calibration')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

AGES = ['0-4','5-9','10-14','15-19','20-24','25-29','30-34','35-39',
        '40-44','45-49','50-54','55-59','60-64','65-69','70+']

## 1. 결과 로드

In [ ]:
result_nm = load_result(OUT / '2019-2020_by_age_NM.json')
result_lbfgs = load_result(OUT / '2019-2020_by_age_LBFGS.json')

print(f"NM       — method={result_nm.method}, NLL={result_nm.nll:.2f}, evals={result_nm.n_evaluations}")
print(f"L-BFGS-B — method={result_lbfgs.method}, NLL={result_lbfgs.nll:.2f}, evals={result_lbfgs.n_evaluations}")

## 2. NLL / Performance 비교

In [ ]:
summary = pl.DataFrame([
    {
        'method':        r.method,
        'success':       r.success,
        'nll_initial':   r.nll_initial,
        'nll_final':     r.nll,
        'nll_improved':  r.nll_initial - r.nll,
        'evaluations':   r.n_evaluations,
        'elapsed_min':   r.elapsed_seconds / 60,
    }
    for r in (result_nm, result_lbfgs)
])
print(summary)

## 3. 핵심 파라미터 비교

In [ ]:
params_df = pl.DataFrame([
    {
        'method':       r.method,
        'beta_h':       r.calibration.beta_h,
        'beta_w':       r.calibration.beta_w,
        'beta_s':       r.calibration.beta_s,
        'beta_o':       r.calibration.beta_o,
        'gamma_report': r.calibration.gamma_report,
        'seas_amp':     r.seasonality_amp,
        'seas_base':    r.seasonality_base,
        'seas_sigma':   r.seasonality_sigma,
        'phi_min':      float(r.calibration.phi.min()),
        'phi_max':      float(r.calibration.phi.max()),
    }
    for r in (result_nm, result_lbfgs)
])
print(params_df)

In [ ]:
# β + γ_report + seasonality bar plot
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))

# β + γ_report
names = ['β_h', 'β_w', 'β_s', 'β_o', 'γ_report']
vals_nm = [result_nm.calibration.beta_h, result_nm.calibration.beta_w,
           result_nm.calibration.beta_s, result_nm.calibration.beta_o,
           result_nm.calibration.gamma_report]
vals_lb = [result_lbfgs.calibration.beta_h, result_lbfgs.calibration.beta_w,
           result_lbfgs.calibration.beta_s, result_lbfgs.calibration.beta_o,
           result_lbfgs.calibration.gamma_report]
x = np.arange(len(names)); width = 0.35
axes[0].bar(x - width/2, vals_nm, width, label='Nelder-Mead', color='C3')
axes[0].bar(x + width/2, vals_lb, width, label='L-BFGS-B', color='C0')
axes[0].set_xticks(x); axes[0].set_xticklabels(names)
axes[0].set_title('β + γ_report')
axes[0].legend(); axes[0].grid(True, alpha=0.3, axis='y')

# Seasonality
names2 = ['amp', 'base', 'sigma/100']
vals_nm2 = [result_nm.seasonality_amp, result_nm.seasonality_base,
            result_nm.seasonality_sigma / 100]
vals_lb2 = [result_lbfgs.seasonality_amp, result_lbfgs.seasonality_base,
            result_lbfgs.seasonality_sigma / 100]
x2 = np.arange(len(names2))
axes[1].bar(x2 - width/2, vals_nm2, width, label='Nelder-Mead', color='C3')
axes[1].bar(x2 + width/2, vals_lb2, width, label='L-BFGS-B', color='C0')
axes[1].set_xticks(x2); axes[1].set_xticklabels(names2)
axes[1].set_title('Seasonality params (sigma scaled by 1/100)')
axes[1].legend(); axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(OUT / '2019-2020_compare_params.png', dpi=120)
plt.show()

In [ ]:
# φ_a 비교
fig, ax = plt.subplots(figsize=(13, 4.5))
x = np.arange(15); width = 0.35
ax.bar(x - width/2, result_nm.calibration.phi, width, label='Nelder-Mead', color='C3')
ax.bar(x + width/2, result_lbfgs.calibration.phi, width, label='L-BFGS-B', color='C0')
ax.axhline(1.0, color='gray', linestyle='--', label='reference (ϕ_25-29=1)')
ax.set_xticks(x); ax.set_xticklabels(AGES, rotation=45)
ax.set_ylabel(r'$\phi_a$')
ax.set_title('Age-specific susceptibility — method comparison')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(OUT / '2019-2020_compare_phi.png', dpi=120)
plt.show()

## 4. ILI 시계열 비교 (overlay)

In [ ]:
inputs = build_aggregated_inputs()
pop_15 = inputs['pop_15'].flatten()
target_age = load_ili_target_by_age('2019-2020', first_peak_only=True, first_peak_end_week=26)
weeks = target_age['week_in_season']

def predict(res):
    disease = DiseaseParameters(
        seasonality_mode=res.seasonality_mode,
        seasonality_amp=res.seasonality_amp,
        seasonality_base=res.seasonality_base,
        seasonality_sigma=res.seasonality_sigma,
    )
    params = ModelParameters(disease=disease).with_calibration(res.calibration)
    seed_by_age = (
        estimate_initial_infected_from_ili(
            '2019-2020', pop_15, gamma_report_assumed=res.gamma_report_assumed,
        )
        if res.use_data_seed else None
    )
    sim = simulate_aggregated(
        params, inputs,
        seed_total=res.seed_total if not res.use_data_seed else 0.0,
        seed_by_age=seed_by_age,
        initial_immunity=res.initial_immunity,
        t_span=(0.0, 364.0),
    )
    S = sim.states[:, IDX_S, :, :].sum(axis=-1)
    daily_inc = -np.diff(S, axis=0)
    return simulation_to_ili_by_age(
        daily_inc, pop_15, res.calibration.gamma_report, n_weeks=52,
    )

pred_nm = predict(result_nm)
pred_lb = predict(result_lbfgs)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8), sharex=True)
for i, ag in enumerate(ILI_AGE_GROUPS):
    ax = axes[i // 4, i % 4]
    ax.plot(weeks, target_age['ili_rates'][ag], 'ko-', markersize=3, label='observed')
    ax.plot(weeks, pred_nm[ag], color='C3', linewidth=2,
            label=f'NM (NLL={result_nm.nll:.0f})')
    ax.plot(weeks, pred_lb[ag], color='C0', linewidth=2, linestyle='--',
            label=f'L-BFGS-B (NLL={result_lbfgs.nll:.0f})')
    ax.axvspan(26, 52, alpha=0.1, color='gray')
    ax.set_title(f'Age {ag}')
    ax.grid(True, alpha=0.3)
    if i % 4 == 0:
        ax.set_ylabel('ILI / 1000')
    if i // 4 == 1:
        ax.set_xlabel('Week')
axes[1, 3].axis('off')
axes[0, 0].legend(loc='upper right', fontsize=8)
fig.suptitle('2019-2020 fit — Nelder-Mead vs L-BFGS-B overlay', fontsize=13)
plt.tight_layout()
plt.savefig(OUT / '2019-2020_compare_fit.png', dpi=120)
plt.show()

## 5. 그룹별 RMSE 비교

In [ ]:
rows = []
for ag in ILI_AGE_GROUPS:
    obs = target_age['ili_rates'][ag]
    w = target_age['weights'][ag]
    mask = w > 0
    rmse_nm = float(np.sqrt(np.mean((obs[mask] - pred_nm[ag][mask]) ** 2)))
    rmse_lb = float(np.sqrt(np.mean((obs[mask] - pred_lb[ag][mask]) ** 2)))
    rows.append({
        'age_group':   ag,
        'obs_peak':    float(np.nanmax(obs)),
        'pred_peak_nm': float(pred_nm[ag].max()),
        'pred_peak_lb': float(pred_lb[ag].max()),
        'rmse_nm':     rmse_nm,
        'rmse_lb':     rmse_lb,
        'better':      'NM' if rmse_nm < rmse_lb else 'L-BFGS-B',
    })
print(pl.DataFrame(rows))

## 6. 결론

**판단 기준**:
- **NLL 낮음** = 더 좋은 fit (수학적). 단 corner solution 주의.
- **파라미터 합리성**: β 가 (0.01, 1.0) 안 정상값인지, φ_a 가 양 끝 (0.3 또는 3.0) 에 몰려있지 않은지.
- **시즌 시작 false peak 없음**: pred_peak 이 observed peak week 근방인지.
- **연령 패턴**: 어린이 ILI 가 높은 것 잘 잡는지 (φ_a 또는 β_school 통해).

**다음 단계** (현재 fit 결과 따라):
- 두 방법 결과 비슷 + 합리적 → 더 안정적인 한 방법으로 다른 시즌 fit
- L-BFGS-B 가 여전히 corner → bounds 더 좁히거나 multi-start
- 둘 다 false peak → seed_e_factor=0, 또는 initial_immunity 상향, 또는 모델 구조 검토